# SH17 YOLOv9c benchmark notebook

Notebook này dùng dataset ở `E:/data/SH17`, tải weights vào `E:/models/yolov9`, lưu checkpoint dưới `E:/models/sh17_yolov9c_runs`, và ghi leaderboard vào `D:/DS-AI/SH17/artifacts/sh17_yolov9c`.

In [ ]:
from pathlib import Path

import pandas as pd

from scripts.sh17_yolov9c_pipeline import (
    append_result,
    build_dataset_manifest,
    build_oversampled_train_manifest,
    ensure_yolov9c_weights,
    expand_experiments,
    load_experiment_config,
    train_one_experiment,
)


In [ ]:
CONFIG_PATH = Path("configs/sh17_yolov9c_experiments.yaml")
cfg = load_experiment_config(CONFIG_PATH)

dataset_manifest = build_dataset_manifest(
    cfg["paths"]["dataset_root"],
    Path(cfg["paths"]["artifact_root"]) / "dataset",
)
weights_path = ensure_yolov9c_weights(cfg["paths"]["weights_dir"])
experiments = expand_experiments(cfg, dataset_manifest)

print("weights:", weights_path)
print("train images:", dataset_manifest.train_count)
print("val images:", dataset_manifest.val_count)


In [ ]:
if any(exp.get("use_oversampled_train_manifest") for exp in experiments):
    oversampled_manifest = build_oversampled_train_manifest(
        train_manifest=dataset_manifest.train_manifest,
        labels_dir=Path(cfg["paths"]["dataset_root"]) / "labels",
        output_path=Path(cfg["paths"]["artifact_root"]) / "dataset" / "train_oversampled.txt",
        minority_class_ids={2, 4, 6, 10, 13, 16},
        repeat_factor=3,
    )
    for exp in experiments:
        if exp.get("use_oversampled_train_manifest"):
            exp["train_manifest_override"] = str(oversampled_manifest)

[(exp["name"], exp.get("train_manifest_override")) for exp in experiments]


In [ ]:
ACTIVE_EXPERIMENTS = [
    "yolov9c_baseline_640",
    "yolov9c_tuned_640",
    "yolov9c_multiscale_960",
    "yolov9c_oversample_minority_960",
]

records = []
for exp in experiments:
    if exp["name"] not in ACTIVE_EXPERIMENTS:
        continue
    record = train_one_experiment(exp, cfg["paths"]["runs_root"])
    append_result(record, cfg["paths"]["artifact_root"])
    records.append(record)

pd.DataFrame(records).sort_values(["map50", "map"], ascending=False)


In [ ]:
leaderboard = pd.read_csv(Path(cfg["paths"]["artifact_root"]) / "leaderboard.csv")
leaderboard.sort_values(["map50", "map"], ascending=False).head(10)
